# BÀI TẬP LỚN 03: TỐI ƯU HÓA SIÊU THAM SỐ VÀ GIẢI THUẬT HUẤN LUYỆN (DIABETES)
**Học phần:** Phát triển hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

**Mục tiêu bài thực hành:**
1. Khảo sát ảnh hưởng của Learning Rate (0.1, 0.01, 0.001, 0.0001) và Batch Size (16, 32, 64, 128).
2. So sánh đối chuẩn 3 giải thuật tối ưu hóa phổ biến: SGD với Momentum, RMSprop, và Adam.
3. Cài đặt cơ chế Dừng sớm (Early Stopping) chống hiện tượng quá khớp (Overfitting).


In [ ]:
# Cell [1]: Nạp dữ liệu và Chuẩn hóa 76 chiều
import os, sys, time, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
import matplotlib.pyplot as plt

candidate_paths = [
    '../data/diabetes.csv',
    'data/diabetes.csv',
    '../../data/diabetes.csv'
]
data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

df = pd.read_csv(data_path)
X_raw = df.drop('Outcome', axis=1).values
y_raw = df['Outcome'].values

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_raw)
np.random.seed(42)
extra_weights = np.random.randn(X_raw.shape[1], 76 - X_poly.shape[1])
X_extra = X_raw @ extra_weights
X_diab = np.hstack([X_poly, X_extra])

scaler = StandardScaler()
X_norm = scaler.fit_transform(X_diab)

X_train, X_test, y_train, y_test = train_test_split(X_norm, y_raw, test_size=0.2, random_state=42, stratify=y_raw)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")


In [ ]:
# Cell [2]: Thử nghiệm đối sánh 3 thuật toán tối ưu hóa: SGD vs RMSprop vs Adam
class DiabetesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(76, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

optimizers_to_test = {
    'SGD + Momentum': lambda m: optim.SGD(m.parameters(), lr=0.01, momentum=0.9),
    'RMSprop': lambda m: optim.RMSprop(m.parameters(), lr=0.001),
    'Adam': lambda m: optim.Adam(m.parameters(), lr=0.001)
}

history = {}
criterion = nn.BCEWithLogitsLoss()
X_tr_t = torch.FloatTensor(X_train)
y_tr_t = torch.FloatTensor(y_train).unsqueeze(1)

for opt_name, opt_func in optimizers_to_test.items():
    torch.manual_seed(42)
    m = DiabetesNet()
    opt = opt_func(m)
    losses = []
    for ep in range(30):
        m.train()
        opt.zero_grad()
        loss = criterion(m(X_tr_t), y_tr_t)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    history[opt_name] = losses
    print(f"[Done] {opt_name}: Final Loss = {losses[-1]:.4f}")


In [ ]:
# Cell [3]: Trực quan hóa tốc độ hội tụ của các thuật toán tối ưu
plt.figure(figsize=(7, 4.5))
for opt_name, losses in history.items():
    plt.plot(range(1, 31), losses, label=opt_name, lw=2)
plt.title('Comparison of Optimizers Convergence on Diabetes Task')
plt.xlabel('Epoch')
plt.ylabel('BCE Loss')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
